In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_7 import BioJepa, BioJepaConfig
from training_v0_7 import create_model, maybe_compile
from config_v0_7 import DataConfig
from evals.evals import EvalContext, run_encoder_evals, run_composer_evals, run_ac_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/v0_7').expanduser()
ref_root = Path('~/data/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir=ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cuda


In [3]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=6,
    heads=4,
    embed_dim=256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=5.699,
    film_linear_multiple=0.6769,
    sim_coeff=50.18,
    std_coeff=25.44,            # sim_coeff * std_to_sim_ratio (0.5069)
    cov_coeff=0.5158,           # sim_coeff * cov_to_sim_ratio (0.01028)
    pert_latent_dim=128,
    pert_mode_dim=64,
    predictor_embed_dim=128,
    predictor_n_layer=4,
    predictor_heads=4,
)


EVAL_BATCH_SIZE = 64

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters()):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters()):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters()):,}')

Student/Teacher: 7,979,650
ACpredictor: 2,565,760
PerturbationComposer: 444,224


### Load Model 

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_evaled_final.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Encoder Training Evals

In [ ]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
pt_eval_results = run_encoder_evals(eval_ctx)



In [ ]:
save_report(pt_eval_results, data_cfg.eval_results_dir / 'encoder_eval_report.json')
pt_eval_results

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Alignment Training Eval

In [ ]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
align_eval_results = run_composer_evals(align_eval_ctx)

save_report(align_eval_results, data_cfg.eval_results_dir / 'composer_eval_report.json')
align_eval_results

In [ ]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()

### ACPredictor Eval

In [6]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_evaled_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
keys = decoder.load_state_dict(decoder_sd)
keys

<All keys matched successfully>

In [7]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED,
})
full_eval_results = run_ac_evals(eval_ctx)

save_report(full_eval_results, data_cfg.eval_results_dir / 'ac_eval_report.json')
full_eval_results

Using cuda
found 135 shards for split test


Running test inference:   0%|                                                              | 0/5400 [00:00<?, ?it/s]

Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


/home/ubuntu/miniconda3/envs/general/lib/python3.14/site-packages/torch/_inductor/select_algorithm.py:4628: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  current_out_size = out_base.storage().size()
E0717 17:09:23.561000 233493 site-packages/torch/_inductor/select_algorithm.py:4888] [0/0] Runtime error during autotuning: 
E0717 17:09:23.561000 233493 site-packages/torch/_inductor/select_algorithm.py:4888] [0/0] No valid triton configs. OutOfMemoryError: out of resource: triton_mm Required: 245760 Hardware limit:232448 Reducing block sizes or `num_stages` may help.. 
E0717 17:09:23.561000 233493 site-packages/torch/_inductor/select_algorithm.py:4888] [0/0] Ignoring this choice.
E0717 17:09:24.391000 233493 site-packages/torch/_inductor/select_algorit

Aggregated 1416 single-pert, 13 multi-pert perturbations, 345600 samples, 135 shards
  adamson: 11 perturbations, 4288 samples
  k562e_raw: 286 perturbations, 46506 samples
  k562gw: 1053 perturbations, 189095 samples
  norman: 10 perturbations, 9472 samples
  rep1e: 287 perturbations, 22838 samples
  sciplex: 54 perturbations, 73401 samples
Cached test inference to /home/ubuntu/data/v0_7/test_inference_cache (135 shards)
expression_prediction: Pearson=0.8999, R2=0.8024, Centroid_acc=0.0090
gene_level_analysis: Dir_acc=0.7031, Top50_acc=0.6080


perturbation_retrieval (dna):   0%|                                                         | 0/200 [00:00<?, ?it/s]E0717 17:47:14.212000 233493 site-packages/torch/_inductor/select_algorithm.py:4888] [1/1] Runtime error during autotuning: 
E0717 17:47:14.212000 233493 site-packages/torch/_inductor/select_algorithm.py:4888] [1/1] No valid triton configs. OutOfMemoryError: out of resource: triton_mm Required: 245760 Hardware limit:232448 Reducing block sizes or `num_stages` may help.. 
E0717 17:47:14.212000 233493 site-packages/torch/_inductor/select_algorithm.py:4888] [1/1] Ignoring this choice.
Autotune Choices Stats:
{"num_choices": 19, "num_triton_choices": 18, "best_kernel": "triton_mm_3706", "best_kernel_desc": "ACC_TYPE='tl.float32', ALLOW_TF32=True, BLOCK_K=32, BLOCK_M=32, BLOCK_N=64, EVEN_K=True, GROUP_M=8, OUT_DTYPE='tl.float32', USE_FAST_ACCUM=False, num_stages=5, num_warps=8", "best_time": 0.0049600000493228436, "best_triton_pos": 0}
AUTOTUNE mm(64x128, 128x128)
strides: [12

perturbation_retrieval (dna): MRR=0.0007


perturbation_retrieval (chemical): 100%|████████████████████████████████████████████| 54/54 [00:19<00:00,  2.81it/s]


perturbation_retrieval (chemical): MRR=0.0577
Loaded dataset gene masks: ['k562e_raw', 'rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']
uncertainty_calibration: ECE=0.3804, Monotonicity=44.44%
Loading KEGG_2026...
  352 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
moa_matching expression: Within=0.8461, Between=0.8467, Gap=-0.0006, Ratio=0.9993x
moa_matching latent: Within=0.8797, Between=0.8754, Gap=0.0044, Ratio=1.0050x
Loaded Norman combo mapping: 132 combos
Loaded Norman single-gene deltas: 105 genes
Loaded Norman GI subtypes: 88 combos
Loaded dataset splits: ['k562e_raw', 'rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']
combination_perturbation: 13 combo perts, 4283 samples, 13 additive baseline, 5 GI-labeled, 13 generalization-classified
dose_response: monotonicity=50.62%, real_mono=48.77%, spearman=0.0201, curve_sim=0.13014456159784865
Saved report to /home/ubuntu/data/v0_7/eval_results/ac_eval_report.json


{'expression_prediction': {'config': {'test_perturbations': 1416,
   'genes': 10000,
   'test_samples': 345600},
  'sample_level': {'mse': 0.3259465692767925,
   'pearson_r_top20': 0.7959666817337253},
  'perturbation_level': {'r2_all_genes': {'mean': 0.8023879072592084,
    'median': 0.8323186933994293},
   'r2_top50_degs': {'mean': 0.5817424602343537, 'median': 0.6118486821651459},
   'mse': {'mean': 0.09867779165506363, 'median': 0.10011516511440277},
   'pearson_all_genes': {'mean': 0.8999035692231804,
    'median': 0.9143694341182709},
   'pearson_delta_all_genes': {'mean': 0.0858811539893205,
    'median': 0.07232259586453438},
   'pearson_top50_degs': {'mean': 0.24476130360192855,
    'median': 0.24145229905843735}},
  'centroid_accuracy': {'accuracy': 0.00904295403165034, 'n_groups': 1327},
  'vs_baseline': {'beat_rate': 0.0, 'n_evaluated': 1416},
  'severity': {'pearson_r': 0.027821185067296028,
   'spearman_r': -0.16684624104102302},
  'error_by_magnitude': {'0-0.25': {'mae':

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()